# Acne Detection using YOLOv8 - Optimized for 70-80% mAP50

This notebook implements **high-accuracy training** for acne detection using YOLOv8 with optimized hyperparameters.

## 🎯 Target
- **Goal**: Achieve 70-80% mAP50
- **Dataset**: Roboflow (acne04_new version 2)
- **Model**: YOLOv8 Medium with multi-scale training


In [ ]:
# Install required packages
%pip install ultralytics opencv-python matplotlib numpy tqdm pandas roboflow


Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Check device availability and configure for Mac M4 GPU
import torch

# Check if MPS (Metal Performance Shaders) is available for Mac M4
if torch.backends.mps.is_available():
    device = 'mps'
    print("✅ GPU (MPS) is available! Using GPU for training.")
    print("   This will significantly speed up training on Mac M4.")
elif torch.cuda.is_available():
    device = 'cuda'
    print("✅ CUDA GPU is available!")
else:
    device = 'cpu'
    print("⚠️ Using CPU (slower). Consider using GPU if available.")

print(f"\nDevice selected: {device}")
print(f"MPS built: {torch.backends.mps.is_built()}")
print(f"PyTorch version: {torch.__version__}")


✅ GPU (MPS) is available! Using GPU for training.
   This will significantly speed up training on Mac M4.

Device selected: mps
MPS built: True
PyTorch version: 2.9.0


In [ ]:
# Import Libraries
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO
import random
import time
import pandas as pd
import yaml


## 1. Download Dataset from Roboflow


In [ ]:
%pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="B0NQSUPe9ENM4O6rTvIW")
project = rf.workspace("nycu-wnnjh").project("acne04_new")
version = project.version(2)
dataset = version.download("yolov8")

print(f"✅ Dataset downloaded to: {dataset.location}")


Note: you may need to restart the kernel to use updated packages.
loading Roboflow workspace...
loading Roboflow project...
✅ Dataset downloaded to: /Users/quangthai/Documents/AI in Bioinfomatics/acne2/Acne04_new-2


## 2. Data Exploration


In [ ]:
# Sử dụng dataset từ Roboflow
data_yaml_path = os.path.join(dataset.location, 'data.yaml')
print(f"✅ Dataset location: {dataset.location}")
print(f"✅ Data YAML path: {data_yaml_path}")

# Kiểm tra cấu trúc dataset
train_path = os.path.join(dataset.location, 'train')
val_path = os.path.join(dataset.location, 'valid')
test_path = os.path.join(dataset.location, 'test')

# Count images and labels
if os.path.exists(train_path):
    train_images = os.listdir(os.path.join(train_path, 'images')) if os.path.exists(os.path.join(train_path, 'images')) else []
    train_labels = os.listdir(os.path.join(train_path, 'labels')) if os.path.exists(os.path.join(train_path, 'labels')) else []
else:
    train_images = []
    train_labels = []

if os.path.exists(val_path):
    val_images = os.listdir(os.path.join(val_path, 'images')) if os.path.exists(os.path.join(val_path, 'images')) else []
    val_labels = os.listdir(os.path.join(val_path, 'labels')) if os.path.exists(os.path.join(val_path, 'labels')) else []
else:
    val_images = []
    val_labels = []

if os.path.exists(test_path):
    test_images = os.listdir(os.path.join(test_path, 'images')) if os.path.exists(os.path.join(test_path, 'images')) else []
    test_labels = os.listdir(os.path.join(test_path, 'labels')) if os.path.exists(os.path.join(test_path, 'labels')) else []
else:
    test_images = []
    test_labels = []

print(f'\n📊 Dataset Statistics:')
print(f'Training images: {len(train_images)}, labels: {len(train_labels)}')
print(f'Validation images: {len(val_images)}, labels: {len(val_labels)}')
print(f'Test images: {len(test_images)}, labels: {len(test_labels)}')

# Đọc data.yaml để xem số classes
if os.path.exists(data_yaml_path):
    with open(data_yaml_path, 'r') as f:
        data_config = yaml.safe_load(f)
    print(f'\n📋 Dataset Config:')
    print(f'   Classes: {data_config.get("nc", "N/A")}')
    print(f'   Class names: {data_config.get("names", "N/A")}')


✅ Dataset location: /Users/quangthai/Documents/AI in Bioinfomatics/acne2/Acne04_new-2
✅ Data YAML path: /Users/quangthai/Documents/AI in Bioinfomatics/acne2/Acne04_new-2/data.yaml

📊 Dataset Statistics:
Training images: 2982, labels: 2982
Validation images: 283, labels: 283
Test images: 142, labels: 142

📋 Dataset Config:
   Classes: 1
   Class names: ['fore']


## 3. Hyperparameter Configuration - Optimized for 70-80% mAP50


In [ ]:
## Hyperparameter Configuration - TỐI ƯU ĐỂ ĐẠT 70-80% mAP50

# === CONFIG: High Accuracy - Đạt 70-80% mAP50 ===
TRAIN_CONFIG = {
    'model': 'yolov8m.pt',      # ⬆️ Medium model - accuracy cao hơn small
    'imgsz': 640,               # ⬆️ Resolution tốt cho small objects
    'batch': 8,                 # ⬇️ Batch nhỏ do model lớn hơn
    'lr0': 0.01,                # Learning rate ổn định
    'lrf': 0.01,                # Final LR thấp để ổn định
    'epochs': 300,              # ⬆️ Nhiều epochs để học tốt
    'patience': 50,             # ⬆️ Patience cao hơn
    
    # === Warmup ===
    'warmup_epochs': 5.0,       # Warmup dài để ổn định
    'warmup_momentum': 0.8,
    'warmup_bias_lr': 0.1,
    
    # === Augmentation - TĂNG để cải thiện accuracy ===
    'degrees': 15.0,            # ⬆️ Rotation tăng
    'shear': 10.0,              # ⬆️ Shear tăng
    'mixup': 0.15,              # ⬆️ Mixup tăng
    'mosaic': 1.0,
    'copy_paste': 0.1,          # ⬆️ BẬT copy-paste
    'translate': 0.2,           # ⬆️ Translation tăng
    
    # === Learning Rate Schedule ===
    'cos_lr': True,
    
    # === Multi-scale BẬT - QUAN TRỌNG cho accuracy ===
    'multi_scale': True,        # ⬆️ BẬT multi-scale
    
    # === Close mosaic ===
    'close_mosaic': 15,         # Tắt mosaic sớm hơn
    
    # === Cache data ===
    'cache': True,
    
    # === Loss weights - Tối ưu cho small objects ===
    'box': 7.5,
    'cls': 0.5,
    'dfl': 1.5,
}

print("✅ Configuration TỐI ƯU ĐỂ ĐẠT 70-80% mAP50:")
print("   📈 Model: yolov8m (lớn hơn, accuracy cao hơn)")
print("   📈 Image size: 640 (tốt hơn cho small objects)")
print("   📈 Learning rate: 0.01 (ổn định)")
print("   📈 Epochs: 300 (nhiều hơn để học tốt)")
print("   📈 Multi-scale: BẬT (cải thiện accuracy)")
print("   📈 Augmentation tăng: degrees, shear, mixup, copy-paste")
print("   📈 Warmup: 5 epochs (ổn định hơn)")
print()
for key, value in TRAIN_CONFIG.items():
    print(f"   {key}: {value}")


✅ Configuration TỐI ƯU ĐỂ ĐẠT 70-80% mAP50:
   📈 Model: yolov8m (lớn hơn, accuracy cao hơn)
   📈 Image size: 640 (tốt hơn cho small objects)
   📈 Learning rate: 0.01 (ổn định)
   📈 Epochs: 300 (nhiều hơn để học tốt)
   📈 Multi-scale: BẬT (cải thiện accuracy)
   📈 Augmentation tăng: degrees, shear, mixup, copy-paste
   📈 Warmup: 5 epochs (ổn định hơn)

   model: yolov8m.pt
   imgsz: 640
   batch: 8
   lr0: 0.01
   lrf: 0.01
   epochs: 300
   patience: 50
   warmup_epochs: 5.0
   warmup_momentum: 0.8
   warmup_bias_lr: 0.1
   degrees: 15.0
   shear: 10.0
   mixup: 0.15
   mosaic: 1.0
   copy_paste: 0.1
   translate: 0.2
   cos_lr: True
   multi_scale: True
   close_mosaic: 15
   cache: True
   box: 7.5
   cls: 0.5
   dfl: 1.5


## 4. Initialize YOLO Model


In [ ]:
# Initialize YOLOv8 model với config đã chọn
model = YOLO(TRAIN_CONFIG['model'])
print(f"✅ Model initialized: {TRAIN_CONFIG['model']}")
print("   Model ready for high-accuracy training!")


✅ Model initialized: yolov8m.pt
   Model ready for high-accuracy training!


## 5. Train the Model - High Accuracy Training

**Note:** 
- Training will use GPU (MPS) on Mac M4 if available
- Optimized for accuracy: Target 70-80% mAP50
- Model: yolov8m with multi-scale training


In [ ]:
# Clear MPS cache before training (if using MPS)
if device == 'mps':
    torch.mps.empty_cache()
    torch.backends.mps.benchmark = False
    print("✅ MPS cache cleared")


✅ MPS cache cleared


In [ ]:
# Track training start time
training_start_time = time.time()

# Sử dụng dataset từ Roboflow
data_yaml_path = os.path.join(dataset.location, 'data.yaml')
print(f"📁 Using dataset: {data_yaml_path}")

# Train với config TỐI ƯU ĐỂ ĐẠT 70-80% mAP50
results = model.train(
    data=data_yaml_path,
    
    # === Epochs & Early Stopping ===
    epochs=TRAIN_CONFIG['epochs'],
    patience=TRAIN_CONFIG['patience'],
    
    # === Image Size ===
    imgsz=TRAIN_CONFIG['imgsz'],
    
    # === Batch Size ===
    batch=TRAIN_CONFIG['batch'],
    
    # === Learning Rate ===
    lr0=TRAIN_CONFIG['lr0'],
    lrf=TRAIN_CONFIG['lrf'],
    momentum=0.937,
    weight_decay=0.0005,
    
    # === Warmup ===
    warmup_epochs=TRAIN_CONFIG['warmup_epochs'],
    warmup_momentum=TRAIN_CONFIG['warmup_momentum'],
    warmup_bias_lr=TRAIN_CONFIG['warmup_bias_lr'],
    
    # === Augmentation - Tăng để cải thiện accuracy ===
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=TRAIN_CONFIG['degrees'],
    translate=TRAIN_CONFIG.get('translate', 0.1),
    scale=0.5,
    shear=TRAIN_CONFIG['shear'],
    perspective=0.0,
    flipud=0.0,
    fliplr=0.5,
    mosaic=TRAIN_CONFIG['mosaic'],
    mixup=TRAIN_CONFIG['mixup'],
    copy_paste=TRAIN_CONFIG.get('copy_paste', 0.0),
    auto_augment='randaugment',
    erasing=0.4,
    
    # === Loss Weights ===
    box=TRAIN_CONFIG.get('box', 7.5),
    cls=TRAIN_CONFIG.get('cls', 0.5),
    dfl=TRAIN_CONFIG.get('dfl', 1.5),
    
    # === Multi-scale BẬT ===
    multi_scale=TRAIN_CONFIG.get('multi_scale', True),
    
    # === Close Mosaic ===
    close_mosaic=TRAIN_CONFIG.get('close_mosaic', 15),
    
    # === Tối ưu ===
    device=device,
    workers=8,
    amp=True,
    save=True,
    plots=True,
    cache=TRAIN_CONFIG.get('cache', False),
    
    # === Learning Rate Schedule ===
    cos_lr=TRAIN_CONFIG['cos_lr'],
    
    # === Optimizer ===
    optimizer='AdamW',
    
    # === Validation ===
    val=True,
    split='val',
    
    # === Name ===
    name='acne_detection_70_80',
)

training_time = time.time() - training_start_time

print("\n✅ Training completed!")
print(f"⏱️  Thời gian training: {training_time/60:.1f} phút ({training_time/3600:.2f} giờ)")
print(f"📊 Best model: {results.save_dir}/weights/best.pt")
print(f"📊 Last model: {results.save_dir}/weights/last.pt")


📁 Using dataset: /Users/quangthai/Documents/AI in Bioinfomatics/acne2/Acne04_new-2/data.yaml
Ultralytics 8.3.237 🚀 Python-3.11.13 torch-2.9.0 MPS (Apple M4 Pro)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=15, cls=0.5, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/Users/quangthai/Documents/AI in Bioinfomatics/acne2/Acne04_new-2/data.yaml, degrees=15.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=300, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.15, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=True, name=acne_detection_

KeyboardInterrupt: 

In [ ]:
# Monitor tốc độ training và metrics
results_dir = results.save_dir if 'results' in locals() else 'runs/detect/acne_detection_70_80'

if os.path.exists(os.path.join(results_dir, 'results.csv')):
    df = pd.read_csv(os.path.join(results_dir, 'results.csv'))
    
    epochs_completed = len(df)
    
    print(f"📊 Epochs completed: {epochs_completed}")
    if 'training_time' in locals():
        print(f"⚡ Tốc độ: {training_time/epochs_completed:.1f} giây/epoch")
        print(f"⚡ Tốc độ: {60/(training_time/epochs_completed):.1f} epochs/phút")
    
    # Check mAP50 sau 30 epochs
    if len(df) >= 30:
        map50_30 = df.iloc[29]['metrics/mAP50(B)']
        print(f"\n📈 mAP50 sau 30 epochs: {map50_30:.4f} ({map50_30*100:.2f}%)")
        if map50_30 > 0.3:
            print("✅ Tốt! Model đang học tốt trong 30 epochs đầu")
        else:
            print("⚠️ Cảnh báo! mAP50 < 30% sau 30 epochs - cần điều chỉnh")
    
    # Check mAP50 cuối cùng
    if len(df) > 0:
        map50_final = df.iloc[-1]['metrics/mAP50(B)']
        map50_95_final = df.iloc[-1]['metrics/mAP50-95(B)']
        print(f"\n📈 Metrics cuối cùng:")
        print(f"   mAP50: {map50_final:.4f} ({map50_final*100:.2f}%)")
        print(f"   mAP50-95: {map50_95_final:.4f} ({map50_95_final*100:.2f}%)")
        
        if map50_final > 0.7:
            print("🎉 Đạt mục tiêu 70%+ mAP50!")
        elif map50_final > 0.5:
            print("✅ Tốt! Đạt trên 50% mAP50")
        else:
            print("⚠️ Cần cải thiện thêm")
else:
    print(f"⚠️ Results file not found at: {results_dir}")


## 7. Load Best Model


In [ ]:
# Tự động load best model sau khi training
results_dir = results.save_dir if 'results' in locals() else 'runs/detect/acne_detection_70_80'
best_model_path = os.path.join(results_dir, 'weights', 'best.pt')
last_model_path = os.path.join(results_dir, 'weights', 'last.pt')

# Kiểm tra và load best model
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)
    print(f"✅ Loaded BEST model from: {best_model_path}")
    print("   This model has the highest mAP50-95 score during training.")
elif os.path.exists(last_model_path):
    model = YOLO(last_model_path)
    print(f"⚠️ Best model not found. Loaded LAST model from: {last_model_path}")
else:
    print("⚠️ No saved model found. Using current model.")

# Hiển thị thông tin model
print(f"\nModel info:")
print(f"  - Device: {device}")
print(f"  - Ready for inference")


## 8. Evaluate the Model


In [ ]:
# Evaluate on test set (using GPU if available)
data_yaml_path = os.path.join(dataset.location, 'data.yaml')
metrics = model.val(data=data_yaml_path, split='test', device=device)
print(f"\n📊 Test Results:")
print(f"   mAP50: {metrics.box.map50:.4f} ({metrics.box.map50*100:.2f}%)")
print(f"   mAP50-95: {metrics.box.map:.4f} ({metrics.box.map*100:.2f}%)")
print(f"   Precision: {metrics.box.mp:.4f} ({metrics.box.mp*100:.2f}%)")
print(f"   Recall: {metrics.box.mr:.4f} ({metrics.box.mr*100:.2f}%)")

# Check if target achieved
if metrics.box.map50 > 0.7:
    print("\n🎉 SUCCESS! Đạt mục tiêu 70%+ mAP50!")
elif metrics.box.map50 > 0.5:
    print("\n✅ Tốt! Đạt trên 50% mAP50")
    print("   Có thể cải thiện thêm bằng cách tăng epochs hoặc sử dụng yolov8l")
else:
    print("\n⚠️ Cần cải thiện thêm")
    print("   Thử: tăng epochs, sử dụng yolov8l, hoặc tăng image size")


## Summary

### ✅ Optimizations Applied for 70-80% mAP50:
- **Model**: yolov8m (medium - better accuracy than small)
- **Image Size**: 640px (good for small objects)
- **Batch Size**: 8 (appropriate for medium model)
- **Learning Rate**: 0.01 (stable)
- **Epochs**: 300 (more epochs for better learning)
- **Multi-scale**: Enabled (improves accuracy)
- **Augmentation**: Enhanced (degrees, shear, mixup, copy-paste)
- **Warmup**: 5 epochs (stable start)
- **Cache**: Enabled (faster I/O)

### 📊 Expected Results:
- **Target**: 70-80% mAP50
- **Training Time**: ~4-6 hours (depending on GPU)
- **Best Model**: Saved in `runs/detect/acne_detection_70_80/weights/best.pt`

### 📁 Model Location:
The trained model is saved in `runs/detect/acne_detection_70_80/weights/best.pt`
